In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
card = pd.read_csv(r"C:\Data science work\python\Credit Card Fraud Detection\creditcard.csv")

In [4]:
print("\n shape of the data : \n ",card.shape)
print("\n data type : \n",card.dtypes )
print("\n number of missing value in the data : \n", card.isnull().sum())
print("\n number of duplicates : \n", card.duplicated().sum())
print("\n data info: \n",card.info())
print("\n describe data :\n",card.describe(include='all'))


 shape of the data : 
  (284807, 31)

 data type : 
 Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

 number of missing value in the data : 
 Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27     

In [5]:
card = card.drop_duplicates() # we got around  1081 number of duplicates which need to be removed 

print("print the updated shape of the data",card.shape) 

print the updated shape of the data (283726, 31)


In [6]:
card.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
card["Class"].value_counts() # checking the imbalnce of the data data 
                             # here 0 means normal transaction and 1 means Fraud detection transactions 
                             # Fraud detection is a highly imbalanced classification problem.

Class
0    283253
1       473
Name: count, dtype: int64

Separate Features and Target

In [8]:
x = card.drop("Class", axis = 1) # X → all transaction features
y = card["Class"] # y → fraud label (0 = normal, 1 = fraud)

Train-Test Split

In [9]:
from sklearn.model_selection import train_test_split



x_train, x_test, y_train, y_test = train_test_split(
    x,y,
    test_size = 0.2, # → 20% for testing and 80% for Training set 
    random_state = 42, # → ensures same split every time
    stratify = y # → keeps fraud ratio same in train and test
)

print("Training set:",x_train.shape)
print("testing set : ",x_test.shape) 

Training set: (226980, 30)
testing set :  (56746, 30)


Feature Scaling

ML models work better when features are on the same scale.

example: These values are very different → model struggles.

Amount = 25,000

V1 = -1.2

V2 = 0.8 

What StandardScaler does: It converts all values to:

mean = 0

standard deviation = 1


In [10]:

from sklearn.preprocessing import StandardScaler 

scaler = StandardScaler()

x_train_scaled =scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

# Now all features are normalized.

Handle Imbalanced Data (SMOTE)

Fraud cases are very few.

So we use SMOTE (Synthetic Minority Oversampling Technique).

Creates artificial fraud samples

Balances fraud and normal data

Prevents model bias

In [11]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42 )

x_train_balanced, y_train_balanced = smote.fit_resample(
    x_train_scaled, y_train
)

print("Before smote :\n", y_train.value_counts())

print("\nafter smote : \n", y_train_balanced.value_counts())

# now will see = Normal ≈ Fraud [Now the model learns fraud patterns properly.]

# Balanced dataset → better fraud learning.

Before smote :
 Class
0    226602
1       378
Name: count, dtype: int64

after smote : 
 Class
0    226602
1    226602
Name: count, dtype: int64


C:\Users\muham\Downloads\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\muham\Downloads\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\muham\Downloads\anaconda\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\muham\Downloads\anaconda\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~

Model Training & Evaluation

In [12]:
#Classification algorithms
#Fraud evaluation tools
#Performance metrics

from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

Train Logistic Regression Model

In [13]:
log_model = LogisticRegression(max_iter = 1000)

log_model.fit(x_train_balanced, y_train_balanced)

LogisticRegression(max_iter=1000)

predict the test data 

Uses unseen test data.

Simulates real-world fraud detection

In [14]:
y_pred_log = log_model.predict(x_test_scaled)

Evaluate Logistic Regression

This tells us:

Precision → how many detected frauds were actually fraud (0.05 ↦ But produces huge false alarms)

Recall  → how many frauds were caught (Recall 0.87 ↦ Catches most fraud)

F1-score → balance between precision and recall

1507 false alarms ↦ Very bad for real banks

Confusion Matrix → actual vs predicted

ROC-AUC → overall fraud detection 



In [15]:
print("Logistic Regression Report : ")
print(classification_report(y_test, y_pred_log))

print("\n Confusion Metrix :")
print(confusion_matrix(y_test, y_pred_log))

print("\n ROC AUC Score : ", roc_auc_score(y_test, y_pred_log))



Logistic Regression Report : 
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56651
           1       0.05      0.87      0.10        95

    accuracy                           0.97     56746
   macro avg       0.53      0.92      0.54     56746
weighted avg       1.00      0.97      0.98     56746


 Confusion Metrix :
[[55144  1507]
 [   12    83]]

 ROC AUC Score :  0.9235413691772989


Train Random Forest Model

Random Forest is powerful for fraud detection

Handles complex patterns

Works well on tabular data

n_estimators=100 → 100 decision trees

n_jobs=-1 → use all CPU cores

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=1000,
    random_state=42,
    n_jobs= -1
)

rf_model.fit(x_train_balanced, y_train_balanced)


RandomForestClassifier(n_estimators=1000, n_jobs=-1, random_state=42)

Predict using Random Forest

In [17]:
y_pred_rf = rf_model.predict(x_test_scaled)

Evaluate Random Forest

Fraud (Class 1):

Precision = 0.89 ↦ When model says "Fraud", it's correct 89% of the time

Recall    = 0.75 ↦ Model catches 75% of all frauds

F1-score  = 0.81 

Only 24 frauds missed ↦	Very strong

Only 9 false alarms   ↦ Very low

Confusion Matrix:

[[56642     9]   → Normal predicted as Normal/Fraud

[   24    71]]  → Fraud predicted as Normal/Fraud


In [18]:
print("Random Fores Repot :\n", classification_report(y_test, y_pred_rf))


print("\n Confusion Metrix :\n", confusion_matrix(y_test, y_pred_rf))

print("\n ROC AUC Score :", roc_auc_score(y_test, y_pred_rf))


Random Fores Repot :
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.89      0.75      0.81        95

    accuracy                           1.00     56746
   macro avg       0.94      0.87      0.91     56746
weighted avg       1.00      1.00      1.00     56746


 Confusion Metrix :
 [[56642     9]
 [   24    71]]

 ROC AUC Score : 0.8736047768005211



Final Verdict

Logistic Regression

Too many false positives → bad user experience

Random Forest( our best model)

Balanced, accurate, strong fraud detector

Random Forest is your final production model.

# Model Saving 

Save Random Forest model as .pkl

Save scaler


In [19]:
# this will make sure the file save to the folder we want to be saved in --- IGNORE ---

import os 
os.chdir(r"C:\Data science work\python\Credit Card Fraud Detection") # change the directory to the folder where we want to save the file

In [20]:

import joblib

# Save the Model
joblib.dump(rf_model, "fraud_model.pkl")
print("model saved")

# save the scalar
joblib.dump(scaler, "scaler.pkl")
print("scaler saved")

model saved
scaler saved
